In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructField,
    StructType,
    StringType,
    IntegerType,
    DoubleType,
    LongType
)


#  Define source schema

schema = StructType([
    StructField("InvoiceNo", StringType(), True),
    StructField("StockCode", StringType(), True),
    StructField("Description", StringType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("InvoiceDate", StringType(), True),
    StructField("UnitPrice", DoubleType(), True),


    StructField("CustomerID", DoubleType(), True),

    StructField("Country", StringType(), True)
])


# 3. Read Bronze CSV


input_path = "/Volumes/data_migration_project/default/online_retail/online_retail.csv"

df = (
    spark.read
    .option("header", True)
    .option("encoding", "ISO-8859-1")
    .schema(schema)
    .csv(input_path)
)


print("========== SOURCE DATA ==========")

print("Initial record count:", df.count())

df.printSchema()

df.show(10, truncate=False)



# 4. Remove exact duplicates

df = df.dropDuplicates()

print("After removing duplicates:", df.count())


# 5. Convert data types

df = (
    df
    .withColumn(
        "InvoiceDate",
        F.to_timestamp(
            "InvoiceDate",
            "yyyy-MM-dd HH:mm:ss"
        )
    )

    .withColumn(
        "CustomerID",
        F.col("CustomerID").cast(LongType())
    )
)


# 6. Create Revenue

df = df.withColumn(
    "Revenue",
    F.col("Quantity") * F.col("UnitPrice")
)


# 7. Identify transaction type

df = df.withColumn(
    "TransactionType",
    F.when(
        F.col("InvoiceNo").startswith("C"),
        "RETURN"
    )
    .otherwise("SALE")
)


# 8. Create date columns

df = (
    df
    .withColumn("Year", F.year("InvoiceDate"))
    .withColumn("Month", F.month("InvoiceDate"))
    .withColumn("Day", F.dayofmonth("InvoiceDate"))
)


# 9. Data quality rules

valid_condition = (
    F.col("Quantity").isNotNull()
    & F.col("UnitPrice").isNotNull()
    & F.col("InvoiceDate").isNotNull()
    & (F.col("Quantity") != 0)
    & (F.col("UnitPrice") > 0)
)


valid_df = df.filter(valid_condition)

rejected_df = df.filter(~valid_condition)


print("========== DATA QUALITY ==========")

print("Valid records:", valid_df.count())

print("Rejected records:", rejected_df.count())


# 10. Inspect valid data

print("========== VALID DATA ==========")

valid_df.show(10, truncate=False)


#  10.1 Write Sliver Locally

sliver_path='/Volumes/data_migration_project/default/online_retail/silver/retail'

(
    valid_df
    .write
    .mode('overwrite')
    .partitionBy('Year','Month')
    .parquet(sliver_path)
 )


# 11. Inspect rejected data

print("========== REJECTED DATA ==========")

rejected_df.show(10, truncate=False)

# 11.1  write reject records locally

rejected_path='/Volumes/data_migration_project/default/online_retail/rejected/retail'

(
    rejected_df
    .write
    .mode('overwrite')
    .parquet(rejected_path)
)


# 12. Transaction summary

print("========== TRANSACTION SUMMARY ==========")

df.groupBy("TransactionType").count().show()


# 13. Revenue summary

print("========== REVENUE SUMMARY ==========")

valid_df.select(
    F.sum("Revenue").alias("TotalRevenue")
).show()


# 14. Date range

print("========== DATE RANGE ==========")

df.select(
    F.min("InvoiceDate").alias("MinDate"),
    F.max("InvoiceDate").alias("MaxDate")
).show()


In [0]:
from pyspark.sql import SparkSession

# Ensure we have an active Spark session
try:
    # Test if the session is active
    spark.version
except:
    # Session was stopped - create a new one
    spark = SparkSession.builder.master('local[*]').getOrCreate()

test_df=spark.read.parquet(
    '/Volumes/data_migration_project/default/online_retail/rejected/retail'
)

test_df.printSchema()
print('rejected count: ', test_df.count())

test_df.show(10,truncate=False)

In [0]:
print('Sliver Path data')
test_df=spark.read.parquet(
    '/Volumes/data_migration_project/default/online_retail/silver/retail'
)

test_df.printSchema()
print('Sliver Count: ',test_df.count())

test_df.show(10,truncate=False)